# Deploying and Using MedImageParse3D Model for Inference using Online Endpoints
This example illustrates how to deploy MedImageParse3D, a state-of-the-art 3D volumetric segmentation model tailored for biomedical imaging. For this Notebook, we use Python 3.10, AzureML v2.

### Task
The primary task is 3D volumetric semantic segmentation, where the goal is to identify and label specific regions within a 3D medical volume (CT, MRI, PET) based on their semantic meaning using a submitted volume and a text prompt.
 
### Model
MedImageParse3D is powered by a BoltzFormer transformer-based architecture, fine-tuned for 3D segmentation tasks on extensive biomedical volumetric datasets. It is designed to excel in handling complex segmentation challenges across diverse 3D imaging modalities. 

### Inference data
For this demonstration, we will use 3D NIfTI volumes (e.g., CT or MRI scans) and perform volumetric segmentation of anatomical structures and pathologies.

### Outline
1. Setup pre-requisites
2. Pick a model to deploy
3. Deploy the model to an online endpoint
4. Test the endpoint
5. Clean up resources - delete the endpoint


## 1. Setup pre-requisites
* Install [Azure ML Client library for Python](https://learn.microsoft.com/en-us/python/api/overview/azure/ai-ml-readme?view=azure-python)
* Connect to AzureML Workspace and authenticate.

In [ ]:
from azure.ai.ml import MLClient
from azure.identity import (
    DefaultAzureCredential,
    InteractiveBrowserCredential,
)
from azure.ai.ml.entities import (
    ManagedOnlineEndpoint,
    ManagedOnlineDeployment,
)

try:
    credential = DefaultAzureCredential()
except Exception as ex:
    credential = InteractiveBrowserCredential()

## 2. Pick a model to deploy

In this example, we use the `MedImageParse3D` model. If you have opened this notebook for a different model, replace the model name accordingly.


In [ ]:
# The models are available in the AzureML system registry, "azureml"
registry_ml_client = MLClient(
    credential,
    registry_name="azureml",
)
model = registry_ml_client.models.get(name="MedImageParse3D", label="latest")
ml_client = MLClient.from_config(credential)


## 3. Deploy the model to an online endpoint for real time inference
Online endpoints give a durable REST API that can be used to integrate with applications that need to use the model.

The steps below show how to deploy an endpoint programmatically. You can skip the steps in this section if you just want to test an existing endpoint. 

In [ ]:
import random
import string

endpoint_name = "MedImageParse3D"

# Creating a unique endpoint name by including a random suffix
allowed_chars = string.ascii_lowercase + string.digits
endpoint_suffix = "".join(random.choice(allowed_chars) for x in range(5))
endpoint_name = f"{endpoint_name}-{endpoint_suffix}"

print(f"Endpoint name: {endpoint_name}")


In [ ]:
endpoint = ManagedOnlineEndpoint(name=endpoint_name)
endpoint = ml_client.online_endpoints.begin_create_or_update(endpoint).result()

In [ ]:
from azure.ai.ml.entities import OnlineRequestSettings, ProbeSettings

deployment_name = "medimageparse3d-v1"
deployment_package = ManagedOnlineDeployment(
    name=deployment_name,
    endpoint_name=endpoint_name,
    model=model,
    instance_type="Standard_NC40ads_H100_v5",
    instance_count=1,
    request_settings=OnlineRequestSettings(request_timeout_ms=180000),  # 3D volumes need more time
    app_insights_enabled=True,
    liveness_probe=ProbeSettings(initial_delay=600),
)


In [ ]:
_ = ml_client.online_deployments.begin_create_or_update(deployment_package).result()

## 4 Test the endpoint - base64 encoded NIfTI volume and text

The MedImageParse3D endpoint accepts a base64-encoded NIfTI volume (`.nii.gz`) and a text prompt describing the structures to segment. The model returns 3D segmentation masks along with text labels for each segmented structure.

Please follow the data download instructions in the main [README](../../README.md) to download the sample data for this notebook.


In [ ]:
# If you skipped the programmatic deployment step, and just want to test the endpoint that you already have, you can uncomment the strings below and add your endpoint and deployment names.
# endpoint_name = ""
# deployment_name = ""

In [ ]:
import json
import base64
import matplotlib.pyplot as plt
import numpy as np
import nibabel as nib
import os


def read_volume(volume_path):
    """Read a NIfTI volume file as raw bytes for base64 encoding."""
    with open(volume_path, "rb") as f:
        return f.read()


data_root = "/home/azureuser/data/healthcare-ai"  # Change to the location you downloaded the data
sample_volume = os.path.join(data_root, "medimageparse3d-volumes", "example_volume.nii.gz")

data = {
    "input_data": {
        "columns": ["image", "text"],
        "index": [0],
        "data": [
            [
                base64.encodebytes(read_volume(sample_volume)).decode("utf-8"),
                "tumor",
            ]
        ],
    }
}
data_json = json.dumps(data)

# Create request json
request_file_name = "sample_request_data.json"
with open(request_file_name, "w") as request_file:
    json.dump(data, request_file)


In [ ]:
def decode_json_to_array(json_encoded):
    """Decode an image pixel data array in JSON.
    Return image pixel data as an array.
    """
    array_metadata = json.loads(json_encoded)
    base64_encoded = array_metadata["data"]
    shape = tuple(array_metadata["shape"])
    dtype = np.dtype(array_metadata["dtype"])
    array_bytes = base64.b64decode(base64_encoded)
    array = np.frombuffer(array_bytes, dtype=dtype).reshape(shape)
    return array


def plot_3d_segmentation_slices(volume, segmentation_masks, labels=None, slice_idx=None, axis=2):
    """Plot a central slice from the 3D volume with segmentation mask overlays.

    Args:
        volume: 3D numpy array
        segmentation_masks: list of 3D numpy arrays (one per prompt/label)
        labels: list of label strings for each mask
        slice_idx: index of the slice to display (default: middle slice)
        axis: axis along which to take the slice (0=sagittal, 1=coronal, 2=axial)
    """
    if slice_idx is None:
        slice_idx = volume.shape[axis] // 2

    vol_slice = np.take(volume, slice_idx, axis=axis)
    vol_slice = ((vol_slice - vol_slice.min()) / (vol_slice.max() - vol_slice.min() + 1e-8) * 255).astype(np.uint8)

    n_masks = len(segmentation_masks)
    fig, ax = plt.subplots(1, n_masks + 1, figsize=(5 * (n_masks + 1), 5))
    if n_masks == 0:
        ax = [ax]
    ax[0].imshow(vol_slice, cmap="gray")
    ax[0].set_title(f"Original (slice {slice_idx})")
    ax[0].axis("off")

    for i, mask in enumerate(segmentation_masks):
        mask_slice = np.take(mask, min(slice_idx, mask.shape[axis] - 1), axis=axis)
        ax[i + 1].imshow(vol_slice, cmap="gray")
        title = labels[i] if labels and i < len(labels) else f"Mask {i+1}"
        ax[i + 1].set_title(title)
        overlay = np.zeros((*mask_slice.shape, 4), dtype=np.float32)
        overlay[mask_slice > 0] = [1.0, 0.0, 0.0, 0.6]
        ax[i + 1].imshow(overlay)
        ax[i + 1].axis("off")
    plt.tight_layout()
    plt.show()


In [ ]:
# Score the sample volume using the online endpoint
response = ml_client.online_endpoints.invoke(
    endpoint_name=endpoint_name,
    deployment_name=deployment_name,
    request_file=request_file_name,
)
result_list = json.loads(response)

# Extract segmentation masks and text labels from the response
image_features_str = result_list[0]["image_features"]
image_features = decode_json_to_array(image_features_str)
text_features = result_list[0].get("text_features", [])

# Split stacked masks: 4D (N, D, H, W) → list of 3D masks
if image_features.ndim == 4:
    masks = [image_features[i] for i in range(image_features.shape[0])]
elif image_features.ndim == 3:
    masks = [image_features]
else:
    masks = [image_features]

print(f"Volume shape: {volume_data.shape}")
print(f"Segmentation masks: {len(masks)} mask(s), shape: {masks[0].shape}")
print(f"Labels: {text_features}")

# Load volume for visualization and plot segmentation masks with labels
nii = nib.load(sample_volume)
volume_data = nii.get_fdata()
plot_3d_segmentation_slices(volume_data, masks, labels=text_features)


## 5. Clean up resources - delete the online endpoint
Don't forget to delete the online endpoint, else you will leave the billing meter running for the compute used by the endpoint.

In [ ]:
ml_client.online_endpoints.begin_delete(name=endpoint_name).wait()